### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_discussion import QuinexDiscussion
from quinex_application.quinex_visualisation import QuinexVisualisation

c:\Users\m.gorres\AppData\Local\miniforge3\envs\quinex_application\lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
discussion = QuinexDiscussion("../data/visualisation_files/regex/entity_patterns.json", "../data/visualisation_files/regex/property_patterns.json")
visualisation = QuinexVisualisation()

### **Load Results**

In [3]:
df_quinex_metadata = discussion.load_df("../data/quinex/df_quinex_metadata.parquet")
df_quinex = discussion.load_df("../data/quinex/df_quinex.parquet")

📂 Loaded DataFrame from '../data/quinex/df_quinex_metadata.parquet' with 31653 rows
📂 Loaded DataFrame from '../data/quinex/df_quinex.parquet' with 3244165 rows


In [ ]:
df_quinex["property"].value_counts().iloc[50:100]

### **Filter data**

In [5]:
df = df_quinex.copy()
entity_regex_map = discussion.entity_regex_map
property_regex_map = discussion.property_regex_map

In [ ]:
property_regex_map = {
    "Temperature": property_regex_map["Temperature"]}
property_regex_map

In [ ]:
entity_regex_map = {"PV Plant": entity_regex_map["PV Plant"], "Wind Farm": entity_regex_map["Wind Farm"], "Solar Power Plant": entity_regex_map["Solar Power Plant"]}
entity_regex_map

In [7]:
df["entity_mapping"] = df["entity"].apply(lambda x: discussion.map_entity(x, entity_regex_map))
label_to_base = discussion.create_label_to_base(entity_regex_map)
df["entity_base_mapping"] = df["entity_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["entity_mapping"].notna()]

In [8]:
df["property_mapping"] = df["property"].apply(lambda x: discussion.map_entity(x, property_regex_map))
label_to_base = discussion.create_label_to_base(property_regex_map)
df["property_base_mapping"] = df["property_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["property_mapping"].notna()]

In [7]:
df["property_base_mapping"].value_counts()

property_base_mapping
Cost             132477
Efficiency        69571
Emission          63200
Capacity          50439
Load              34154
Lifetime          17464
Temperature        7574
Wind               7400
LPSP               3237
Power Density      2461
Name: count, dtype: int64

In [9]:
df[df["entity_mapping"] == "Sodium Battery"]["entity"].value_counts()

entity
sodium sulphur battery                        8
sodium-sulfur battery                         4
Sodium–sulfur battery (NaS)                   4
sodium-sulfur batteries                       4
sodium-nickel-chloride (NaNiCl2) batteries    3
                                             ..
sodium sulphur batteries                      1
Sodium nickel chloride batteries              1
Sodium Sulfur (NaS) battery                   1
sodium-nickel-chloride battery (NaN-iCl2)     1
Na-ion batteries (NIBs)                       1
Name: count, Length: 61, dtype: Int64

In [14]:
len(df_quinex)

3244165

In [ ]:
df[df["property_mapping"] == "Operating Temperature"]["property"].value_counts()

In [7]:
df.to_parquet("../data/visualisation_files/df_technologies.parquet")

### **Filter results**

In [ ]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_capacity.parquet")

In [ ]:
technologies_df["property_mapping"].value_counts()

In [ ]:
technologies_df[technologies_df["property_mapping"] == "Capital Cost"]["property"].value_counts().head(50)

In [ ]:
technologies_df[(technologies_df["property_mapping"] == "CO2 emission") & (technologies_df["entity"] == "Berlin")].head(2)

### **Postprocess Units**

In [ ]:
import pandas as pd
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")
technologies_df = technologies_df[technologies_df["property_mapping"] == "Installed Capacity"]
def dicts_to_tuples(units):
    return [(u.get('text', ''), u.get('exponent', None), u.get('unit_uri', ''), u.get('year', None)) 
            for u in units]

technologies_df['unit_normalized'] = technologies_df['unit_normalized'].apply(dicts_to_tuples)

In [ ]:
technologies_df["entity_base_mapping"].value_counts()

In [ ]:
print(technologies_df["unit_normalized"].value_counts())
print("--------------------------------------------------------------------------------")
exclude_values = [
    [(None, None, 'http://qudt.org/vocab/unit/UNITLESS', None)],
    [('%', 1.0, 'http://qudt.org/vocab/unit/PERCENT', None)], 
]
technologies_df = technologies_df[~technologies_df["unit_normalized"].isin(exclude_values)]
print(technologies_df["unit_normalized"].value_counts().head(10))

In [ ]:
target_unit = [('GW', 1.0, 'http://qudt.org/vocab/unit/MegaW', None)]
#target_unit = [("$", 1.0, "http://qudt.org/vocab/currency/USD", None), ("kW", -1.0, "http://qudt.org/vocab/unit/KiloW", None)]
technologies_df['value_converted'] = technologies_df.apply(
   lambda row: visualisation.convert_row(row, target_unit), axis=1
   )

In [ ]:
technologies_df["unit_normalized"].value_counts()

In [ ]:
technologies_df = technologies_df[technologies_df["value_converted"].notna()]
technologies_df = technologies_df.reset_index(drop=True)
technologies_df.to_excel("../data/visualisation_files/unit_converted/df_technologies_capacity_gw.xlsx")